This notebook is served as the code of Appendix C.3 "Parameter selection for EXP3.M". It consists of the following sections:
1. comparison of $\gamma$ where base arms range in $[0,0.1]$;
2. comparison of $\gamma$ where base arms range in $[0.3,0.4]$;
3. summary of code in 1.;
4. summary of code in 2.;
5. plot of the results.

In [2]:
import numpy as np
import pandas as pd
import os
import time
from tqdm import tqdm
import random 

GLOBAL_SEED = 2026


# -------- Dependent Rounding (for EXP3.M) --------
def dep_round(p, k):
    p = np.array(p, dtype=np.float64).copy()
    n = len(p)
    while True:
        undecided = np.where((p > 1e-8) & (p < 1 - 1e-8))[0]
        if len(undecided) < 2: 
            break
        i, j = undecided[:2]
        alpha = min(1 - p[i], p[j])
        beta = min(p[i], 1 - p[j])
        total = alpha + beta
        if total == 0: 
            break
        prob = beta / total
        if np.random.rand() < prob:
            p[i] += alpha
            p[j] -= alpha
        else:
            p[i] -= beta
            p[j] += beta
        p[i] = np.clip(p[i], 0, 1)
        p[j] = np.clip(p[j], 0, 1)
    
    selected = [i for i in range(n) if p[i] >= 0.999]
    if len(selected) != k:
        sorted_indices = np.argsort(p)[::-1][:k]
        selected = sorted_indices.tolist()
    return selected

# -------- EXP3_M Algorithm --------
class EXP3_M:
    def __init__(self, m, k, gamma):
        self.m = m
        self.k = k
        self.gamma = gamma
        self.weights = np.ones(m)
        self.t = 1

    def select_arms(self):
        self._clip_weights()
        total = np.sum(self.weights_clipped) + 1e-8  
        prob = self.k * ((1 - self.gamma) * self.weights_clipped / total + self.gamma / self.m)
        prob = np.clip(prob, 0, 1) 
        return dep_round(prob, self.k)

    def update(self, selected, rewards):
        self._clip_weights()
        total = np.sum(self.weights_clipped) + 1e-8
        prob = self.k * ((1 - self.gamma) * self.weights_clipped / total + self.gamma / self.m)
        prob = np.clip(prob, 1e-8, 1.0)  

        hat_reward = np.zeros(self.m)
        for arm in selected:
            hat_reward[arm] += rewards[selected.index(arm)] / prob[arm]
        
        update_factor = (self.k * self.gamma / self.m) * hat_reward
        update_factor = np.clip(update_factor, -50, 50)  

        mask = ~np.isin(np.arange(self.m), self.S0)
        self.weights[mask] *= np.exp(update_factor[mask])
        self.weights = np.clip(self.weights, 1e-100, 1e100)  
        self.t += 1

    def _clip_weights(self):
        sum_weights = np.sum(self.weights)
        threshold_ratio = (1/self.k - self.gamma/self.m)/(1 - self.gamma)
        threshold_val = threshold_ratio * sum_weights
        sorted_weights = np.sort(self.weights)[::-1]

        if sorted_weights[0] >= threshold_val:
            low = 0
            high = len(sorted_weights)
            best_alpha = 0.0
            
            while low < high:
                mid = (low + high) // 2
                sum_rest = sum_weights - np.sum(sorted_weights[:mid])
                denominator = mid*(1 - threshold_ratio)
                
                if denominator <= 0:
                    high = mid
                    continue
                    
                candidate = (threshold_ratio * sum_rest) / denominator
                if mid > 0 and candidate > sorted_weights[mid-1]:
                    low = mid + 1
                else:
                    best_alpha = candidate
                    high = mid
            
            alpha = best_alpha
            self.S0 = np.where(self.weights >= alpha)[0]
            self.weights_clipped = np.minimum(self.weights, alpha)
        else:
            self.S0 = np.array([], dtype=int)
            self.weights_clipped = self.weights.copy()

In [2]:
# -------- Bandit Class --------
class PerturbedBernoulliBandit:
    def __init__(self, mu, noise_scale):
        self.base_mu = np.clip(mu, 0, 0.1)
        self.noise_scale = noise_scale

    def pull(self):
        perturbed_mu = np.clip(
            self.base_mu + np.random.normal(0, self.noise_scale),
            0, 0.1
        )
        return np.random.binomial(1, perturbed_mu)


# -------- Experiment Runner --------
class ExperimentRunner:
    def __init__(self, noise_scale):
        self.noise_scale = noise_scale
        self.datasets = {
            'synthetic_30arms': np.random.uniform(0, 0.1, 30).tolist(),  
        }
        self.param_grid = {
            'EXP3_M': {'k': [10, 20], 'gamma': [0.001, 0.01, 0.1]},
        }
        self.data_dir = "results"
        os.makedirs(self.data_dir, exist_ok=True)

    def run(self, n_sim, T):
        conv_data = []
        for dataset, mus in self.datasets.items():
            sorted_mus = sorted(mus, reverse=True)
            for algo in ['EXP3_M']:
                params_list = self._gen_params(algo)
                for params in params_list:
                    k = params['k']
                    gamma = params['gamma']
                    best_exp = sum(sorted_mus[:k])
                    for sim_id in tqdm(range(n_sim), desc=f"{dataset}-{algo}-k{k}-gamma{gamma}"):
                        sim_seed = GLOBAL_SEED + sim_id * 1000 + hash((algo, k)) % 10_000_000
                        random.seed(sim_seed)
                        np.random.seed(sim_seed)
                        
                        bandits = [PerturbedBernoulliBandit(mu, self.noise_scale) for mu in mus]
                        agent = self._init_algo(algo, len(mus), params)
                        cumulative_time = 0.0 
                        current_regret = 0.0
                        for t in range(T):
                            step_start = time.perf_counter()
                            arms = agent.select_arms()
                            rewards = [bandits[i].pull() for i in arms]
                            agent.update(arms, rewards)
                            step_time = time.perf_counter() - step_start
                            cumulative_time += step_time    
                            exp_reward = sum(bandits[i].base_mu for i in arms)
                            current_regret += (best_exp - exp_reward)
                            record = {
                                'dataset': dataset,
                                'algorithm': algo,
                                'gamma': gamma,
                                'k': k,
                                't': t + 1,
                                'regret': current_regret,
                                'cumulative_time': cumulative_time,
                                'noise': self.noise_scale,
                                'sim_id': sim_id
                            }
                            conv_data.append(record)        
        return pd.DataFrame(conv_data)

    def save_data(self, conv_df):
        conv_file = f"{self.data_dir}/l_conv_gamma.csv"
        conv_df.to_csv(conv_file, index=False)

    def _init_algo(self, algo, m, params):  
        if algo == 'EXP3_M':
            return EXP3_M(m=m, k=params['k'], gamma=params['gamma'])
        
    def _gen_params(self, algo):
        params = []
        if algo == 'EXP3_M':  
            for k in self.param_grid[algo]['k']:
                for g in self.param_grid[algo]['gamma']:
                    params.append({'k': k, 'gamma': g})
        return params

# -------- Main --------
if __name__ == "__main__":
    exp = ExperimentRunner(noise_scale=0)
    n_sim, T = 10, 100000  
    conv_df = exp.run(n_sim, T)
    exp.save_data(conv_df)

synthetic_30arms-EXP3_M-k20-gamma0.1: 100%|██████████| 10/10 [16:44<00:00, 100.42s/it]


In [3]:
# -------- Bandit Class --------
class PerturbedBernoulliBandit2:
    def __init__(self, mu, noise_scale):
        self.base_mu = np.clip(mu, 0.3, 0.4)
        self.noise_scale = noise_scale

    def pull(self):
        perturbed_mu = np.clip(
            self.base_mu + np.random.normal(0, self.noise_scale),
            0.3, 0.4
        )
        return np.random.binomial(1, perturbed_mu)

# -------- Experiment Runner --------
class ExperimentRunner2:
    def __init__(self, noise_scale):
        self.noise_scale = noise_scale
        self.datasets = {
            'synthetic_30arms': np.random.uniform(0.3, 0.4, 30).tolist(),  
        }
        self.param_grid = {
            'EXP3_M': {'k': [10, 20], 'gamma': [0.001, 0.01, 0.1]},
        }
        self.data_dir = "results"
        os.makedirs(self.data_dir, exist_ok=True)

    def run(self, n_sim, T):
        conv_data = []
        for dataset, mus in self.datasets.items():
            sorted_mus = sorted(mus, reverse=True)
            for algo in ['EXP3_M']:
                params_list = self._gen_params(algo)
                for params in params_list:
                    k = params['k']
                    gamma = params['gamma']
                    best_exp = sum(sorted_mus[:k])
                    for sim_id in tqdm(range(n_sim), desc=f"{dataset}-{algo}-k{k}-gamma{gamma}"):
                        bandits = [PerturbedBernoulliBandit2(mu, self.noise_scale) for mu in mus]
                        agent = self._init_algo(algo, len(mus), params)
                        cumulative_time = 0.0 
                        current_regret = 0.0
                        for t in range(T):
                            step_start = time.perf_counter()
                            arms = agent.select_arms()
                            rewards = [bandits[i].pull() for i in arms]
                            agent.update(arms, rewards)
                            step_time = time.perf_counter() - step_start
                            cumulative_time += step_time    
                            exp_reward = sum(bandits[i].base_mu for i in arms)
                            current_regret += (best_exp - exp_reward)
                            record = {
                                'dataset': dataset,
                                'algorithm': algo,
                                'gamma': gamma,
                                'k': k,
                                't': t + 1,
                                'regret': current_regret,
                                'cumulative_time': cumulative_time,
                                'noise': self.noise_scale,
                                'sim_id': sim_id
                            }
                            conv_data.append(record)        
        return pd.DataFrame(conv_data)

    def save_data(self, conv_df):
        conv_file = f"{self.data_dir}/s_conv_gamma.csv"
        conv_df.to_csv(conv_file, index=False)

    def _init_algo(self, algo, m, params):  
        if algo == 'EXP3_M':
            return EXP3_M(m=m, k=params['k'], gamma=params['gamma'])
        
    def _gen_params(self, algo):
        params = []
        if algo == 'EXP3_M':  
            for k in self.param_grid[algo]['k']:
                for g in self.param_grid[algo]['gamma']:
                    params.append({'k': k, 'gamma': g})
        return params

# -------- Main --------
if __name__ == "__main__":
    exp = ExperimentRunner2(noise_scale=0)
    n_sim, T = 10, 100000  
    conv_df = exp.run(n_sim, T)
    exp.save_data(conv_df)

synthetic_30arms-EXP3_M-k10-gamma0.001:   0%|          | 0/10 [00:00<?, ?it/s]

synthetic_30arms-EXP3_M-k20-gamma0.1: 100%|██████████| 10/10 [10:33<00:00, 63.32s/it]


In [3]:
def process_results_by_gamma(input_path, output_path, target_steps):
    df = pd.read_csv(input_path)
    filtered = df[df['t'].isin(target_steps)]

    grouped = filtered.groupby(['algorithm', 'k', 'gamma', 't']).agg({
        'regret': 'mean',
        'cumulative_time': 'mean'
    }).reset_index()

    result = grouped.melt(
        id_vars=['algorithm', 'k','gamma', 't'],
        value_vars=['regret', 'cumulative_time'],
        var_name='metric',
        value_name='average_value'
    )

    result.to_csv(output_path, index=False)
    print(f"Processed results saved to {output_path}")

if __name__ == "__main__":
    config = {
        'input_file': 'results/l_conv_gamma.csv',
        'output_file': 'results/l_summary_results_gamma.csv',
        'target_steps': [100000] 
    }

    process_results_by_gamma(
        input_path=config['input_file'],
        output_path=config['output_file'],
        target_steps=config['target_steps']
    )


Processed results saved to results/l_summary_results_gamma.csv


In [4]:
def process_results_by_gamma(input_path, output_path, target_steps):
    df = pd.read_csv(input_path)
    filtered = df[df['t'].isin(target_steps)]

    grouped = filtered.groupby(['algorithm', 'k', 'gamma', 't']).agg({
        'regret': 'mean',
        'cumulative_time': 'mean'
    }).reset_index()

    result = grouped.melt(
        id_vars=['algorithm', 'k','gamma', 't'],
        value_vars=['regret', 'cumulative_time'],
        var_name='metric',
        value_name='average_value'
    )

    result.to_csv(output_path, index=False)
    print(f"Processed results saved to {output_path}")

if __name__ == "__main__":
    config = {
        'input_file': 'results/s_conv_gamma.csv',
        'output_file': 'results/s_summary_results_gamma.csv',
        'target_steps': [100000] 
    }

    process_results_by_gamma(
        input_path=config['input_file'],
        output_path=config['output_file'],
        target_steps=config['target_steps']
    )

Processed results saved to results/s_summary_results_gamma.csv
